# Summary

Using LangGraph evaluation tools, evaluate the CCC rag Bbot.

## Import libraries

In [1]:
import os, sys
import json

from langsmith import wrappers, Client
from pydantic import BaseModel, Field
from openai import OpenAI
from langsmith import traceable

########## Change for rag testing
sys.path.insert(0, "utils")
sys.path.insert(0, "../../interface/utils")
from gcp_tools import download_directory_from_gcs
from authentication import ApiAuthentication

sys.path.insert(0, "../../interface/rag")
import rag_bot as rb


## Get API keys

In [2]:
dot_env_path = "../../data/environment"

# Get creds if needed
if len(dot_env_path) > 0:
    creds = ApiAuthentication(dotenv_path=dot_env_path)

    # LangSmith
    os.environ["LANGCHAIN_TRACING_V2"] = "true"
    os.environ["LANGCHAIN_TRACING"] = "true"
    os.environ["LANGCHAIN_API_KEY"] = creds.apis_configs["LANGCHAIN_API_KEY"]
    # Google
    os.environ["GOOGLE_API_KEY"] = creds.apis_configs["GOOGLE_API_KEY"]
    # OpenAI
    os.environ["OPEN_API_KEY"] = creds.apis_configs["OPENAI_KEY"]


## Establish a Langchain client and OpenAI wrapper

In [3]:
client = Client()
openai_client = wrappers.wrap_openai(OpenAI(api_key=os.environ["OPEN_API_KEY"]))


## Read in test question examples

In [4]:
data_path = "../../../../Numantic/Archive/ccc-tests/data/qa_ipeds_data"
file_name = "ipeds_qa_1.json"
# json_str = json.dumps(qandas)

with open(os.path.join(data_path, file_name), "r") as file:
    examples = json.load(file)

# type(examples)


## Create a Langchain example dataset

In [5]:
inputs = [{"question": input_prompt} for input_prompt, _ in examples]
outputs = [{"answer": output_answer} for _, output_answer in examples]

# # Programmatically create a dataset in LangSmith
# dataset = client.create_dataset(
# 	dataset_name = "Sample dataset",
# 	description = "A sample dataset in LangSmith."
# )
#
# # Add examples to the dataset
# client.create_examples(inputs=inputs, outputs=outputs, dataset_id=dataset.id)


## Create target testing function using the CCC RAG bot

In [6]:
# Instantiate a CCC bot
bot = rb.CCCPolicyAssistant(chroma_collection_name="crawl_docs-vai-2",
                            chat_bot_verbose=False,
                            dot_env_path=dot_env_path)

# Create a target function
@traceable()
def target(inputs: dict) -> dict:

    # Get the bot response
    bot.show_conversation(input_message=inputs["question"])

    # Combine into a single response
    ai_response = "{}".format(bot.ai_response)

    # retrieved_urls = ["- [{}]({})\n".format(up[0], up[1]) for up in bot.retrieved_urls]
    # retrieved_urls = list(set(retrieved_urls))
    #
    # # Create a single string of retrieved URLs
    # res_phrase = ""
    # if len(retrieved_urls) > 0:
    #     res_phrase = "\n\nThese references might be useful: \n{}".format(" ".join(retrieved_urls))
    #
    # # Combine into a single response
    # ai_response = "{} {}".format(bot.ai_response, res_phrase)

    # Add query result response
    if len(bot.query_data_result) > 0:
        ai_response = "{}\n{}".format(ai_response, bot.query_data_result)

    return { "response": ai_response }



Downloaded 7553f5e6-0521-4d13-82c7-7af306bfe444/data_level0.bin to data/local_chromadb/7553f5e6-0521-4d13-82c7-7af306bfe444/data_level0.bin
Downloaded 7553f5e6-0521-4d13-82c7-7af306bfe444/header.bin to data/local_chromadb/7553f5e6-0521-4d13-82c7-7af306bfe444/header.bin
Downloaded 7553f5e6-0521-4d13-82c7-7af306bfe444/index_metadata.pickle to data/local_chromadb/7553f5e6-0521-4d13-82c7-7af306bfe444/index_metadata.pickle
Downloaded 7553f5e6-0521-4d13-82c7-7af306bfe444/length.bin to data/local_chromadb/7553f5e6-0521-4d13-82c7-7af306bfe444/length.bin
Downloaded 7553f5e6-0521-4d13-82c7-7af306bfe444/link_lists.bin to data/local_chromadb/7553f5e6-0521-4d13-82c7-7af306bfe444/link_lists.bin
Downloaded chroma.sqlite3 to data/local_chromadb/chroma.sqlite3


In [9]:
# Test target function
# inputs[0]
check = target(inputs[2])
check


{'response': 'I do not have access to enrollment data for California community colleges. However, I can use a search engine to try to find this information for you. What search query would you like me to use?\n'}

## Define evaluator

In [7]:
# Define instructions for the LLM judge evaluator
instructions = """Evaluate Student Answer against Ground Truth for conceptual similarity and classify true or false:
- False: No conceptual match and similarity
- True: Most or full conceptual match and similarity
- Key criteria: Concept should match, not exact wording.
"""

# Define output schema for the LLM judge
class Grade(BaseModel):
    score: bool = Field(description="Boolean that indicates whether the response is accurate relative to the reference answer")

# Define LLM judge that grades the accuracy of the response relative to reference output
def accuracy(outputs: dict, reference_outputs: dict) -> bool:
    response = openai_client.beta.chat.completions.parse(
    model="gpt-4o-mini",
    messages=[
      { "role": "system", "content": instructions },
      { "role": "user", "content": f"""Ground Truth answer: {reference_outputs["answer"]};
      Student's Answer: {outputs["response"]}"""
    }],
    response_format=Grade
    );
    return response.choices[0].message.parsed.score



## Run test

In [8]:
# After running the evaluation, a link will be provided to view the results in langsmith
experiment_results = client.evaluate(
    target,
    data = "Sample dataset",
    evaluators = [
        accuracy,
        # can add multiple evaluators here
    ],
    experiment_prefix = "first-eval-in-langsmith",
    max_concurrency = 1,
)


/opt/anaconda3/envs/ccc-polasst/lib/python3.9/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


View the evaluation results for experiment: 'first-eval-in-langsmith-ad3f9ed3' at:
https://smith.langchain.com/o/522c4165-fa56-4a3e-93ef-53c25a91a9c2/datasets/4c8643c9-2cf9-43fd-a6de-dcbe972397a4/compare?selectedSessions=29ddf34e-b709-4d37-bb8b-e80d05be06b5




10it [00:24,  2.41s/it]
